# Cross-Validation & Hold-Out Testing
**An Empirical Study of Model Ranking Stability**

Reliable model evaluation is essential in machine learning. Two commonly used approaches are hold-out testing and k-fold cross-validation, but their effectiveness can vary depending on dataset size and model characteristics.

This study compares both evaluation strategies using multiple classification algorithms on the Diabetes Health Indicators dataset.  This dataset is derived from the UCI dataset here: https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators. The analysis is performed on both the full dataset and a reduced sample to investigate how evaluation methodology influences model ranking stability.

In [ ]:
# Package imports
import pandas as pd

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

**Experimental Design**

Five classification models will be evaluated:

* k-Nearest Neighbours (default parameters)
* k-Nearest Neighbours (k = 100)
* Decision Tree (default parameters)
* Decision Tree (max depth = 10)
* Gaussian Naive Bayes

Two evaluation methodologies will be compared:

1. 5-Fold Cross-Validation
2. Hold-Out Testing using a 2:1 train-test split

To examine the impact of dataset size, experiments will be performed on both the full dataset and a random sample containing 3,000 observations.

***
## Evaluation on the Full Dataset

Loading the dataset into a Python notebook

In [ ]:
df = pd.read_csv("diabetes_70k.csv")
print(df.shape)
df.head()

y = df.pop("class").values
X = df.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

(70692, 22)


Ranking the following models in terms of accuracy using cross-validation:

> <font color='darkgreen'>**1:**</font> K-Nearest Neighbour :  default parameters

In [ ]:
knn_default = KNeighborsClassifier()

acc_knn_default = cross_val_score(knn_default, X_scaled, y, cv=10).mean()

> <font color='darkgreen'>**2:**</font> K-Nearest Neighbour :  k = 100

In [ ]:
knn_100 = KNeighborsClassifier(n_neighbors=100)

acc_knn_100 = cross_val_score(knn_100, X_scaled, y, cv=10).mean()

> <font color='darkgreen'>**3:**</font> Decision Tree :  default parameters

In [ ]:
dt_default = DecisionTreeClassifier(random_state=0)

acc_dt_default = cross_val_score(dt_default, X_scaled, y, cv=10).mean()

> <font color='darkgreen'>**4:**</font> Decision Tree : max depth = 10

In [ ]:
dt_depth10 = DecisionTreeClassifier(max_depth=10, random_state=0)

acc_dt_depth10 = cross_val_score(dt_depth10, X_scaled, y, cv=10).mean()

> <font color='darkgreen'>**5:**</font> Gaussian Naive Bayes: default parameters

In [ ]:
gnb = GaussianNB()

acc_gnb = cross_val_score(gnb, X_scaled, y, cv=10).mean()

In [ ]:
results = pd.Series({
    "kNN (default)": acc_knn_default,
    "kNN (k=100)": acc_knn_100,
    "Decision Tree (default)": acc_dt_default,
    "Decision Tree (max_depth=10)": acc_dt_depth10,
    "Gaussian Naive Bayes": acc_gnb
}).sort_values(ascending=False)

results

kNN (k=100)                     0.738910
Decision Tree (max_depth=10)    0.735840
Gaussian Naive Bayes            0.716771
kNN (default)                   0.707973
Decision Tree (default)         0.652931
dtype: float64

**The stability of this ranking**

There is also evidence for very high ranking stability based on cross-validation. The most accurate models scored almost identical accuracy percentages, with the k-NN (k = 100) method outperforming other methods slightly (73.89%) followed by the Decision Tree constrained by the maximum depth of decision (73.58%). The second level of accuracy was reached by the Gaussian Naive Bayes and k-NN methods, whereas the unconstrained Decision Tree ranked last.

Cross-validation ensures that no single division between training and testing sets may significantly affect model performance, thus yielding more accurate estimates of generalisation performance. Therefore, since the dataset is large (about 70,000 data instances), rankings produced by this evaluation technique would be rather stable upon further analysis.

***
## Evaluation on a Reduced Dataset

Repeating the test above but using hold-out testing (2:1 split). 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=1/3, random_state=0)

> <font color='darkgreen'>**1:**</font> K-Nearest Neighbour :  default parameters

In [ ]:
knn_default = KNeighborsClassifier()
knn_default.fit(X_train, y_train)

acc_knn_default_ht = knn_default.score(X_test, y_test)

> <font color='darkgreen'>**2:**</font> K-Nearest Neighbour :  k = 100

In [ ]:
knn_100 = KNeighborsClassifier(n_neighbors=100)
knn_100.fit(X_train, y_train)

acc_knn_100_ht = knn_100.score(X_test, y_test)

> <font color='darkgreen'>**3:**</font> Decision Tree :  default parameters

In [ ]:
dt_default = DecisionTreeClassifier(random_state=0)
dt_default.fit(X_train, y_train)

acc_dt_default_ht = dt_default.score(X_test, y_test)

> <font color='darkgreen'>**4:**</font> Decision Tree : max depth = 10

In [ ]:
dt_depth10 = DecisionTreeClassifier(max_depth=10, random_state=0)
dt_depth10.fit(X_train, y_train)

acc_dt_depth10_ht = dt_depth10.score(X_test, y_test)

> <font color='darkgreen'>**5:**</font> Gaussian Naive Bayes: default parameters

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

acc_gnb_ht = gnb.score(X_test, y_test)

In [ ]:
ho_results = pd.Series({
    "kNN (default)": acc_knn_default_ht,
    "kNN (k=100)": acc_knn_100_ht,
    "Decision Tree (default)": acc_dt_default_ht,
    "Decision Tree (max_depth=10)": acc_dt_depth10_ht,
    "Gaussian Naive Bayes": acc_gnb_ht
}).sort_values(ascending=False)

ho_results

kNN (k=100)                     0.743931
Decision Tree (max_depth=10)    0.737566
Gaussian Naive Bayes            0.720336
kNN (default)                   0.711467
Decision Tree (default)         0.654049
dtype: float64

**The stability of the ranking**

Similar model ranking was observed in hold-out validation results as in cross-validation. k-NN with k equal to 100 yielded the best performance in terms of accuracy, which amounted to 74.39%. The second place was taken by the Decision Tree with the maximum depth of 10 (73.76%). Gaussian Naive Bayes and k-NN without parameters came third and fourth, respectively, whereas the worst performance was yielded by the Decision Tree with no limitations.

Even though hold-out validation is more vulnerable to train-test split, in this particular instance, the ranking did not change, indicating that the use of a sufficiently big sample resulted in the performance ranking that was similar to that based on cross-validation.

Nonetheless, unlike cross-validation, the results of hold-out validation are not robust enough since they depend solely on the specific train-test split. "Different train-test partitions could result in slightly different accuracy estimates and model rankings, which means that cross-validation is a better approach to comparing model performance.

***
## Comparative Analysis

Reducing the dataset to a random sample of size 3,000 and repeating the two evaluations above. 

In [ ]:
df_full = pd.read_csv("diabetes_70k.csv")

df_small = df_full.sample(n=3000, random_state=1)

y_small = df_small.pop("class").values
X_small = df_small.values

scaler_small = StandardScaler()
X_small_scaled = scaler_small.fit_transform(X_small)

models = {
    "kNN (default)": KNeighborsClassifier(),
    "kNN (k=100)": KNeighborsClassifier(n_neighbors=100),
    "Decision Tree (default)": DecisionTreeClassifier(random_state=0),
    "Decision Tree (max_depth=10)": DecisionTreeClassifier(max_depth=10, random_state=0),
    "Gaussian Naive Bayes": GaussianNB()
}

Cross-Validation

In [ ]:
cv_results_small = {}

for name, model in models.items():
    scores = cross_val_score(model, X_small_scaled, y_small, cv=10)
    cv_results_small[name] = scores.mean()

cv_small_ranked = pd.Series(cv_results_small).sort_values(ascending=False)
cv_small_ranked

kNN (k=100)                     0.730667
Gaussian Naive Bayes            0.716667
kNN (default)                   0.711333
Decision Tree (max_depth=10)    0.686333
Decision Tree (default)         0.646333
dtype: float64

Hold-out Testing

In [ ]:
X_tr, X_ts, y_tr, y_ts = train_test_split(X_small_scaled, y_small, test_size=1/3, random_state=0)

In [ ]:
ht_results_small = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    ht_results_small[name] = model.score(X_ts, y_ts)

ht_small_ranked = pd.Series(ht_results_small).sort_values(ascending=False)
ht_small_ranked

kNN (k=100)                     0.727
Gaussian Naive Bayes            0.700
kNN (default)                   0.685
Decision Tree (max_depth=10)    0.683
Decision Tree (default)         0.675
dtype: float64

**Comparative Analysis of Evaluation Techniques**

In terms of the trimmed dataset, the two evaluation techniques both concluded that k-NN (with k = 100) was the best performing model and Gaussian Naive Bayes was the second best-performing model. However, there is an increase in difference in the accuracy measures of the poorly performing models. Notably, the difference in accuracy measure of the Decision Trees shrinks with the use of hold-out evaluation.

It is evident from the above observation that trimming the dataset increases uncertainty in model evaluation.

A summary table :
| Dataset Size | Evaluation Method | Most Accurate Model | Accuracy |
| ------------ | ----------------- | ------------------- | -------- |
| 70,000       | Cross-Validation  | k-NN (k=100)        | 0.7389   |
| 70,000       | Hold-Out          | k-NN (k=100)        | 0.7439   |
| 3,000        | Cross-Validation  | k-NN (k=100)        | 0.7307   |
| 3,000        | Hold-Out          | k-NN (k=100)        | 0.7270   |


***

**What do these evaluations tell us about the relative merits of cross-validation and hold-out testing?**

First of all, it needs to be pointed out that the two methods of estimating the performance of predictive models yield almost identical results when there is enough data at one's disposal to build a reliable model. The results have shown that on the full data set the ranking of models and their estimates of accuracy are completely identical. Thus, we conclude that hold-out testing can be an alternative to cross-validation in some cases.

On the other hand, cross-validation appears superior when working with a smaller dataset due to several reasons. First, it yields slightly different estimates for models' performance even when the models under consideration remain the same, which means that in the case of hold-out testing the ranking may change depending on random factors.

To conclude, we have found out that although both hold-out testing and cross-validation may be equally good when it comes to using a large amount of data, the latter proves to be more advantageous when there is no sufficient amount of information for building models.